# 02 - Agent-Based Model Simulation (Revised)

This notebook runs the revised ABM simulation framework that fixes:
1. **Model collapse** — dispatches to Models 1-5 (not just 2 hazard functions)
2. **Exposure window** — 336h (14-day) rolling window with Hawkes re-sharing dynamics
3. **Cross-cluster covariate** — correctly computed from agent adoption history
4. **Hazard-to-probability** — uses `P = 1 - exp(-h)` instead of `P = min(1, h)`

Pipeline:
1. Load data and fitted Cox models (1-5)
2. Compute semantic clustering for cross-cluster covariate
3. Fit Hawkes process for re-sharing dynamics
4. Build simulation config from fitted models
5. Run baseline and quarantine counterfactual scenarios
6. Compare diffusion curves and first-adoption distributions
7. Validate co-adoption patterns against empirical data
8. Hawkes model comparison and simulation check

In [ ]:
%matplotlib inline
import json
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import networkx as nx
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import asdict

from conspiracy_analysis import BOT_SCORE_THRESHOLD, EXPOSURE_WINDOW as PACKAGE_EXPOSURE_WINDOW
from conspiracy_analysis.data.loader import anonymized_graph_path, load_graph_and_tweets
from conspiracy_analysis.config import (
    apply_conspiracy_config_to_graph,
    get_included_conspiracies,
)
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.analysis.semantic import (
    assign_clusters,
    load_clustering_result,
)
from conspiracy_analysis.analysis.statistics import compute_coadoption_matrix
from conspiracy_analysis.simulation.config import (
    build_simulation_config_from_fitted_models,
    override_baselines_to_linear,
)
from conspiracy_analysis.simulation.counterfactuals import compute_entry_times
from conspiracy_analysis.simulation.scenarios import (
    baseline,
    quarantine,
    no_temporal_effects,
    content_moderation,
)
from conspiracy_analysis.simulation.runner import (
    run_scenario,
    run_comparison,
)
from conspiracy_analysis.simulation.evaluation import (
    compute_diffusion_curves,
    compute_first_adoption_distribution,
    compare_scenarios,
    compute_empirical_comparison,
)
from conspiracy_analysis.simulation.provenance import (
    atomic_joblib_dump,
    atomic_json_dump,
    canonicalize_json_value,
    collect_git_state,
    collect_package_versions,
    sha256_file,
)


In [ ]:
from pathlib import Path

INTERMEDIATE_DIR = Path("../intermediate_files")
INTERMEDIATE_DIR.mkdir(exist_ok=True)
SIMULATION_FIGURE_DIR = Path("../figures/simulations")
SIMULATION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path('..').resolve()
initial_git_state_path = os.environ.get('SIMULATION_INITIAL_GIT_STATE_PATH')
if initial_git_state_path:
    state_path = Path(initial_git_state_path)
    if not state_path.exists():
        raise FileNotFoundError(f'Initial git state file is missing: {state_path}')
    with state_path.open('r', encoding='utf-8') as handle:
        INITIAL_GIT_STATE = json.load(handle)
    required_state_fields = {
        'source_commit', 'initial_worktree_clean', 'initial_worktree_state'
    }
    missing_state_fields = required_state_fields.difference(INITIAL_GIT_STATE)
    if missing_state_fields:
        raise ValueError(
            f'Initial git state is missing fields: {sorted(missing_state_fields)}'
        )
    if not isinstance(INITIAL_GIT_STATE['initial_worktree_clean'], bool):
        raise TypeError('initial_worktree_clean must be a boolean')
    if not isinstance(INITIAL_GIT_STATE['initial_worktree_state'], list):
        raise TypeError('initial_worktree_state must be a list')
    if INITIAL_GIT_STATE['initial_worktree_clean'] != (
        len(INITIAL_GIT_STATE['initial_worktree_state']) == 0
    ):
        raise ValueError('Initial worktree clean flag and status lines disagree')
    current_commit = collect_git_state(REPO_ROOT)['source_commit']
    if INITIAL_GIT_STATE['source_commit'] != current_commit:
        raise ValueError(
            'Source commit changed between provenance capture and notebook execution'
        )
else:
    INITIAL_GIT_STATE = collect_git_state(REPO_ROOT)

def intermediate_path(filename):
    return INTERMEDIATE_DIR / filename

def summarize_run_metric(values):
    values = np.asarray(values, dtype=float)
    return {
        'median': float(np.median(values)),
        'mean': float(np.mean(values)),
        'p2_5': float(np.percentile(values, 2.5)),
        'p97_5': float(np.percentile(values, 97.5)),
    }

validation_metrics = {}


In [ ]:
# Publication style (applies rcParams globally)
from conspiracy_analysis.visualization import set_nhb_style
set_nhb_style()

In [ ]:
# ============================================================
# CONFIGURATION — All tunable parameters in one place
# ============================================================

# --- Graph ---
USE_SYNTHETIC = False  # True: simulate on a synthetic Watts-Strogatz network
WS_K = 6              # Watts-Strogatz: each node connected to k nearest neighbors
WS_P = 0.1            # Watts-Strogatz: rewiring probability

# --- Simulation ---
STEPS = 25000            # Number of hourly time steps at most
RUNS = 100              # Independent replications per scenario
SIM_N_JOBS = 128        # Parallel workers per scenario
SEED_FRACTION = 0.01   # Fraction of nodes seeded per conspiracy at entry
EXPOSURE_WINDOW = float(PACKAGE_EXPOSURE_WINDOW)  # 14 days or 336 h, matching Cox training
EQUAL_START = True     # True: all conspiracies seeded at t=1; False: seeded at empirical first-appearance hour

# --- Quarantine counterfactuals (hours) ---
# Literature anchors: 4h = Bak-Coleman et al. 2022 (Nat Hum Behav) circuit-breaker;
# 12h = Twitter COVID first-strike read-only lock; 24h = cross-platform first-tier
# suspension; 168h = Twitter 4th-strike preceding tier / YouTube 7-day freeze.
T_QUARANTINE_4H   = 4
T_QUARANTINE_12H  = 12
T_QUARANTINE_24H  = 24
T_QUARANTINE_168H = 168

# --- Content moderation counterfactuals ---
MODERATION_RATES = [0.2, 0.4, 0.6, 0.8, 0.95]


## 1. Load Data and Fitted Models

In [ ]:
# Load the graph from the published public data archive.
G, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')
assert nx.number_of_selfloops(G) == 0, (
    'Public graph contains self loops. Use the published public data archive.'
)
G = apply_conspiracy_config_to_graph(G)

# Retain the full configured graph only for provenance counts.
# Agent degree is calculated from the final human simulation graph.
G_full_undirected = G.to_undirected(as_view=False)

# Remove non human nodes for simulation
nodes_to_remove = [
    node for node, data in G.nodes(data=True)
    if not passes_bot_filter(G, node, BOT_SCORE_THRESHOLD, 'HUMAN')
]
G.remove_nodes_from(nodes_to_remove)

# Keep only the giant component
giant_nodes = max(nx.connected_components(G.to_undirected()), key=len)
G = G.subgraph(giant_nodes).copy().to_undirected(as_view=False)
G = apply_conspiracy_config_to_graph(G)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

# Keep a reference to the empirical graph for Hawkes fitting and comparison
G_empirical = G

In [ ]:
# Load fitted Cox models (from notebook 01)
cox_models = {}
for i in range(1, 20):
    # For model 2, load model_2a (target conspiracy dummies), not the old model_2
    if i == 2:
        try:
            ctv = joblib.load(intermediate_path('model_2a_cox.pkl'))
            cox_models[i] = ctv
            print("Loaded model_2a_cox.pkl (target conspiracy dummies)")
            continue
        except FileNotFoundError:
            pass
    try:
        ctv = joblib.load(intermediate_path(f'model_{i}_cox.pkl'))
        cox_models[i] = ctv
        print(f"Loaded model_{i}_cox.pkl")
    except FileNotFoundError:
        if i > 2:
            break

G = apply_conspiracy_config_to_graph(G)
CONSPIRACIES = G.graph['conspiracy_cols']
print(f"\nConspiracies ({len(CONSPIRACIES)}): {CONSPIRACIES}")
print(f"Models loaded: {sorted(cox_models.keys())}")

## 2. Compute Semantic Clustering

In [ ]:
CONSPIRACIES

In [ ]:
# Load semantic clustering saved by 01_analysis.ipynb.
clustering_result = load_clustering_result()
cluster_assignments = assign_clusters(clustering_result)

print(f"Optimal clustering: {clustering_result['best_method']} linkage, "
      f"k={clustering_result['best_k']}")
print(f"\nCluster assignments:")
for consp, cluster in sorted(cluster_assignments.items(), key=lambda x: x[1]):
    print(f"  {consp}: cluster {cluster}")


## 3. Build Simulation Config

### 3.5 Fit Hawkes Process for Re-Sharing

After adoption, agents re-share (re-tweet) according to a Hawkes self-exciting process.
Exposure (s7) is computed from neighbors who **shared** within a 14-day rolling window,
matching the Cox training data's `_compute_s_sum()` semantics.

We fit a single **pooled** model (shared parameters across all conspiracies).

In [ ]:
from conspiracy_analysis.models.hawkes_sharing import (
    extract_sharing_sequences,
    fit_hawkes_with_ll,
)

# Compute study end time (max timestamp across all sharing events)
study_end = max(
    t
    for node in G_empirical.nodes()
    for consp in CONSPIRACIES
    for t in G_empirical.nodes[node].get(consp, [])
)
print(f"Study end time: {study_end:.0f}h ({study_end/24:.0f} days)")

# --- Pooled fit (all conspiracies together) ---
sequences, observation_ends = extract_sharing_sequences(G_empirical, CONSPIRACIES, study_end=study_end)
print(f"\nExtracted {len(sequences)} sharing sequences (pooled)")
print(f"  All adopter histories used for fitting: {len(sequences)}")
print(f"  Histories with repeat events: {sum(1 for s in sequences if len(s) >= 2)}")

hawkes_fit = fit_hawkes_with_ll(sequences, observation_ends=observation_ends)
mu, alpha, beta = hawkes_fit.params
print(f"\nHawkes parameters:")
print(f"  mu (baseline rate):       {mu:.6f}")
print(f"  alpha (excitation):       {alpha:.4f}")
print(f"  beta (decay rate):        {beta:.4f}")
print(f"  Mean excitation duration: {1/beta:.1f} hours ({1/beta/24:.1f} days)")
print(f"  Memory cutoff (21/beta):  {21/beta:.1f} hours ({21/beta/24:.1f} days)")
print(f"  Branching ratio (alpha):  {alpha:.4f} {'(subcritical)' if alpha < 1 else '(supercritical!)'}")
print(f"  Log-likelihood:           {hawkes_fit.log_likelihood:.1f}")
print(f"  Selected optimizer start: {hawkes_fit.selected_start}")
print(f"  Converged starts tried:   {hawkes_fit.n_starts}")

hawkes_params = hawkes_fit.params

In [ ]:
from conspiracy_analysis.visualization import plot_hawkes_dynamics

fig = plot_hawkes_dynamics(mu=mu, alpha=alpha, beta=beta)

In [ ]:
# --- Graph Selection ---
if USE_SYNTHETIC:
    n_nodes = G_empirical.number_of_nodes()
    G = nx.watts_strogatz_graph(n_nodes, WS_K, WS_P)
    G = nx.relabel_nodes(G, {i: str(i) for i in G.nodes()})
    for node in G.nodes():
        G.nodes[node]['first_active_time'] = 1
    G.graph['conspiracy_cols'] = CONSPIRACIES
    G = apply_conspiracy_config_to_graph(G)
    print(f"Synthetic Watts-Strogatz graph: n={n_nodes}, k={WS_K}, p={WS_P}")
    print(f"  Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    emp_mean_deg = 2 * G_empirical.number_of_edges() / G_empirical.number_of_nodes()
    print(f"  Mean degree: {2*G.number_of_edges()/G.number_of_nodes():.1f} (empirical: {emp_mean_deg:.1f})")
else:
    print(f"Using empirical graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Compute entry times
entry_times = compute_entry_times(G, CONSPIRACIES, normalize=EQUAL_START)

if not EQUAL_START:
    print("Empirical entry times (hours):")
    for c, t in sorted(entry_times.items(), key=lambda x: x[1]):
        print(f"  {c}: {t:.0f}h ({t/24:.0f} days)")

# Reset all first_active_time to 1 for simulation
for node in G.nodes():
    G.nodes[node]['first_active_time'] = 1

# --- Compute empirical target (total user-conspiracy adoptions) ---
empirical_total_adoptions = 0
empirical_adopters = 0
for node in G_empirical.nodes():
    n_adopted = 0
    for c in CONSPIRACIES:
        acts = G_empirical.nodes[node].get(c, [])
        if acts and not all(np.isnan(a) for a in acts):
            n_adopted += 1
    empirical_total_adoptions += n_adopted
    if n_adopted > 0:
        empirical_adopters += 1

print(f"\nEmpirical adoption target:")
print(f"  Total user-conspiracy adoptions: {empirical_total_adoptions}")
print(f"  Users with any adoption: {empirical_adopters}/{G_empirical.number_of_nodes()}")
print(f"  Mean conspiracies per user: {empirical_total_adoptions / G_empirical.number_of_nodes():.2f}")

# Build simulation config from fitted models
sim_config = build_simulation_config_from_fitted_models(
    cox_models=cox_models,
    conspiracies=CONSPIRACIES,
    cluster_assignments=cluster_assignments,
    entry_times=entry_times,
    seed_fraction=SEED_FRACTION,
    steps=STEPS,
    hawkes_params=hawkes_params,
    exposure_window=EXPOSURE_WINDOW,
    require_peer_exposure=True,
)

# Set target for early stopping
sim_config.target_total_adoptions = empirical_total_adoptions

print(f"\nSimulation config:")
print(f"  Models: {sorted(sim_config.cox_models.keys())}")
print(f"  Conspiracies: {len(sim_config.conspiracies)}")
print(f"  Steps (max): {sim_config.steps}")
print(f"  Target total adoptions: {sim_config.target_total_adoptions}")
print(f"  Seed fraction: {sim_config.seed_fraction}")
print(f"  Max model number: {sim_config.max_model_number}")
print(f"  Hawkes params: mu={hawkes_params[0]:.6f}, alpha={hawkes_params[1]:.4f}, beta={hawkes_params[2]:.4f}")
print(f"  Exposure window: {sim_config.exposure_window} h ({sim_config.exposure_window/24:.0f} days)")
print(f"  Equal start: {EQUAL_START}")

### Diagnostic: Coefficient Comparison Across Models

Why does the independent contagion (Model 1 for everything) outperform the baseline (Models 1-5)?
The Weibull baseline gives ~16x higher h₀ at early τ, but other coefficients may offset this.

In [ ]:
import math

# --- Part 1: Side-by-side coefficient table ---
rows = []
for model_num, spec in sorted(sim_config.cox_models.items()):
    row = {
        'model': f'Model {model_num}',
        'baseline_type': spec.baseline.type,
        'h0(τ=1)': spec.baseline.h0(1.0),
        'h0(τ=8)': spec.baseline.h0(8.0),
        'h0(τ=168)': spec.baseline.h0(168.0),
        'h0(τ=336)': spec.baseline.h0(336.0),
        'beta_degree': spec.beta_degree,
        'beta_cross_cluster': spec.beta_cross_cluster,
    }
    for s7_val in [0, 1, 2, 3, 4]:
        row[f's7_d{s7_val}'] = spec.s7_scores.get(s7_val, 0.0)
    rows.append(row)

df_coeff = pd.DataFrame(rows).set_index('model')
print("=== Coefficient Comparison Across Models ===\n")
print(df_coeff.T.to_string(float_format=lambda x: f'{x:.6f}'))

# --- Part 2: Effective hazard at representative conditions ---
# Use median log_degree from the network
log_degrees = [np.log1p(G.degree(n)) for n in G.nodes()]
med_log_deg = np.median(log_degrees)
print(f"\n\nMedian log(1+degree) = {med_log_deg:.3f}  (degree ≈ {np.exp(med_log_deg)-1:.0f})")

print("\n=== Effective Hazard h = h0(τ) × exp(β·X) ===")
print(f"{'Model':>8} {'τ':>6} {'s7':>4} {'xclust':>6} {'h0':>12} {'exp(βX)':>12} {'h=h0·exp':>12} {'P=1-e^-h':>12}")
print("-" * 80)

for model_num, spec in sorted(sim_config.cox_models.items()):
    for tau in [1, 8, 168, 336]:
        h0 = spec.baseline.h0(float(tau))
        for s7 in [1, 4]:
            for xclust in [0, 1]:
                if model_num == 1 and xclust == 1:
                    continue  # Model 1 has no cross-cluster
                s7_key = min(s7, 4)
                log_ph = (spec.s7_scores.get(s7_key, 0.0)
                          + spec.beta_degree * med_log_deg
                          + spec.beta_cross_cluster * xclust)
                exp_ph = math.exp(log_ph)
                h = h0 * exp_ph
                p = 1 - math.exp(-h)
                print(f"{'M'+str(model_num):>8} {tau:>6} {s7:>4} {xclust:>6} {h0:>12.6e} {exp_ph:>12.4f} {h:>12.6e} {p:>12.6e}")
    print()

# --- Part 3: Ratio analysis ---
print("=== Model 2 vs Model 1: Net Advantage/Disadvantage ===")
print("(ratio > 1 means Model 2 produces higher hazard)\n")
spec1 = sim_config.cox_models[1]
spec2 = sim_config.cox_models[2]

for tau in [1, 8, 24, 168, 336, 800]:
    h0_ratio = spec2.baseline.h0(float(tau)) / spec1.baseline.h0(1.0)  # Model 1 is constant
    for s7 in [1, 4]:
        for xclust in [0, 1]:
            log_ph_1 = spec1.s7_scores.get(min(s7, 4), 0.0) + spec1.beta_degree * med_log_deg
            log_ph_2 = (spec2.s7_scores.get(min(s7, 4), 0.0) + spec2.beta_degree * med_log_deg
                        + spec2.beta_cross_cluster * xclust)
            coeff_ratio = math.exp(log_ph_2 - log_ph_1)
            net_ratio = h0_ratio * coeff_ratio
            print(f"  τ={tau:>4}h, s7={s7}, xclust={xclust}: "
                  f"h0_ratio={h0_ratio:>8.2f}x, coeff_ratio={coeff_ratio:>.4f}x, "
                  f"NET={net_ratio:>8.4f}x {'← M2 wins' if net_ratio > 1 else '← M1 wins'}")
    print()

## 4. Run Baseline Scenario

In [ ]:
scenario_baseline = baseline()
print(f"Running scenario: {scenario_baseline.name}")
print(f"  {scenario_baseline.description}")

results_baseline = run_scenario(
    G, sim_config, scenario_baseline,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_baseline.n_runs} baseline runs")
for run in results_baseline.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")

In [ ]:
# Save baseline results
atomic_joblib_dump(results_baseline, intermediate_path('simulation_baseline_results.pkl'))
print('Baseline results saved.')

In [ ]:
# Configure sim_config for counterfactuals:
# - Run for the same number of steps the baseline took (mean across runs)
# - Disable target stopping (counterfactuals should run the full duration)
counterfactual_steps = results_baseline.mean_steps_completed
sim_config.steps = counterfactual_steps
sim_config.target_total_adoptions = None

print(f"Baseline completed in {counterfactual_steps} steps (mean across {results_baseline.n_runs} runs)")
print(f"All counterfactuals will run for {counterfactual_steps} steps with no early stopping")

## 5. Run Quarantine Counterfactual

After adopting any conspiracy at time T, a user is "quarantined" — they cannot adopt any new conspiracy for the configured quarantine duration. This tests whether a cooling-off period after sharing would slow cross-conspiracy spread.

In [ ]:
scenario_quarantine_168h = quarantine(hours=T_QUARANTINE_168H)
print(f"Running scenario: {scenario_quarantine_168h.name}")
print(f"  {scenario_quarantine_168h.description}")

results_quarantine_168h = run_scenario(
    G, sim_config, scenario_quarantine_168h,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_quarantine_168h.n_runs} quarantine (168h) runs")
for run in results_quarantine_168h.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")


In [ ]:
# Save quarantine results
atomic_joblib_dump(results_quarantine_168h, intermediate_path(f'simulation_quarantine_{T_QUARANTINE_168H}h_results.pkl'))
print('Quarantine (168h) results saved.')


### 5b. Run Additional Quarantine Counterfactuals (4h, 12h, 24h)

Shorter quarantine windows test the dose-response curve at literature-anchored intervention strengths: 4h matches the circuit-breaker tested in Bak-Coleman et al. (2022, *Nat Hum Behav*); 12h matches Twitter's COVID-era first-strike read-only lockout; 24h matches the cross-platform first-tier suspension standard.

In [ ]:
scenario_quarantine_4h = quarantine(hours=T_QUARANTINE_4H)
print(f"Running scenario: {scenario_quarantine_4h.name}")
print(f"  {scenario_quarantine_4h.description}")

results_quarantine_4h = run_scenario(
    G, sim_config, scenario_quarantine_4h,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_quarantine_4h.n_runs} quarantine (4h) runs")
for run in results_quarantine_4h.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")

# Save
atomic_joblib_dump(results_quarantine_4h, intermediate_path('simulation_quarantine_4h_results.pkl'))
print('Quarantine (4h) results saved.')


In [ ]:
scenario_quarantine_24h = quarantine(hours=T_QUARANTINE_24H)
print(f"Running scenario: {scenario_quarantine_24h.name}")
print(f"  {scenario_quarantine_24h.description}")

results_quarantine_24h = run_scenario(
    G, sim_config, scenario_quarantine_24h,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_quarantine_24h.n_runs} quarantine (24h) runs")
for run in results_quarantine_24h.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")

# Save
atomic_joblib_dump(results_quarantine_24h, intermediate_path('simulation_quarantine_24h_results.pkl'))
print('Quarantine (24h) results saved.')

In [ ]:
scenario_quarantine_12h = quarantine(hours=T_QUARANTINE_12H)
print(f"Running scenario: {scenario_quarantine_12h.name}")
print(f"  {scenario_quarantine_12h.description}")

results_quarantine_12h = run_scenario(
    G, sim_config, scenario_quarantine_12h,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_quarantine_12h.n_runs} quarantine (12h) runs")
for run in results_quarantine_12h.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")

# Save
atomic_joblib_dump(results_quarantine_12h, intermediate_path('simulation_quarantine_12h_results.pkl'))
print('Quarantine (12h) results saved.')


### 5c. No Temporal Effects Counterfactual

What if adopting one conspiracy did NOT create a temporary "fever" of increased susceptibility?
Models 2+ have Weibull baselines (k < 1) that start high immediately after a prior adoption and decay over time.
This counterfactual replaces all Weibull baselines with Model 1's constant baseline, removing temporal acceleration while keeping all covariates (peer exposure, degree, cross-cluster, conspiracy dummies) intact.

In [ ]:
# Build modified config: all models use Model 1's constant baseline
sim_config_no_temporal = override_baselines_to_linear(sim_config)

# Verify baselines were replaced
print("No-temporal-effects config baselines:")
for n, spec in sorted(sim_config_no_temporal.cox_models.items()):
    bl = spec.baseline
    print(f"  Model {n}: {bl.type} baseline (slope={bl.slope:.6e})")

scenario_no_temporal = no_temporal_effects()
print(f"\nRunning scenario: {scenario_no_temporal.name}")
print(f"  {scenario_no_temporal.description}")

results_no_temporal = run_scenario(
    G, sim_config_no_temporal, scenario_no_temporal,
    runs=RUNS, n_jobs=SIM_N_JOBS,
)

print(f"\nCompleted {results_no_temporal.n_runs} no-temporal-effects runs")
for run in results_no_temporal.runs:
    n_adopted = len(run.adoption_histories)
    total = sum(len(h) for h in run.adoption_histories.values())
    print(f"  Run {run.run_id}: {n_adopted} nodes adopted, {total} total adoptions")

# Save
atomic_joblib_dump(results_no_temporal, intermediate_path('simulation_no_temporal_effects_results.pkl'))
print('No temporal effects results saved.')

## 6. Compare Diffusion Curves

In [ ]:
# Compute diffusion curves (use actual steps completed, not STEPS cap)
df_baseline = compute_diffusion_curves(results_baseline, steps=counterfactual_steps)
df_quarantine_4h   = compute_diffusion_curves(results_quarantine_4h, steps=counterfactual_steps)
df_quarantine_12h  = compute_diffusion_curves(results_quarantine_12h, steps=counterfactual_steps)
df_quarantine_24h  = compute_diffusion_curves(results_quarantine_24h, steps=counterfactual_steps)
df_quarantine_168h = compute_diffusion_curves(results_quarantine_168h, steps=counterfactual_steps)
df_no_temporal = compute_diffusion_curves(results_no_temporal, steps=counterfactual_steps)

print(f"Diffusion curves computed for {counterfactual_steps} steps")
print(f"Baseline diffusion: {len(df_baseline)} rows")
print(f"Quarantine (4h) diffusion: {len(df_quarantine_4h)} rows")
print(f"Quarantine (12h) diffusion: {len(df_quarantine_12h)} rows")
print(f"Quarantine (24h) diffusion: {len(df_quarantine_24h)} rows")
print(f"Quarantine (168h) diffusion: {len(df_quarantine_168h)} rows")
print(f"No temporal effects diffusion: {len(df_no_temporal)} rows")


In [ ]:
# Plot baseline adoption curves
mean_baseline = df_baseline.groupby(['conspiracy', 't'])['adoption_fraction'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
for consp, group in mean_baseline.groupby('conspiracy'):
    group = group.sort_values('t')
    ax.plot(group['t'], group['adoption_fraction'], label=consp.replace('ConsProb_', ''))
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Adoption Fraction')
ax.set_title('Baseline: Adoption Curves Over Time')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
fig.savefig('../figures/simulations/fig_01_baseline_adoption_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Intervention Effectiveness: Area Under Adoption Curves

The integral of the adoption curve over time (AUC) captures both the magnitude and speed of adoption. Lower AUC = more effective intervention.

In [ ]:
SIM_BOOTSTRAP_N = 2000
SIM_BOOTSTRAP_SEED = 20240517


def bootstrap_mean_ci(values, n_boot=SIM_BOOTSTRAP_N, seed=SIM_BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    estimate = float(np.mean(values)) if values.size else float('nan')
    if values.size == 0:
        return estimate, float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, values.size, size=(n_boot, values.size))
    boot = values[idx].mean(axis=1)
    lower, upper = np.percentile(boot, [2.5, 97.5])
    return estimate, float(lower), float(upper)


def bootstrap_auc_change_ci(
    baseline_values,
    scenario_values,
    intensity=None,
    n_boot=SIM_BOOTSTRAP_N,
    seed=SIM_BOOTSTRAP_SEED,
):
    baseline_values = np.asarray(baseline_values, dtype=float)
    scenario_values = np.asarray(scenario_values, dtype=float)
    if baseline_values.shape != scenario_values.shape:
        raise ValueError('Baseline and scenario arrays must have matching shapes.')
    if baseline_values.size == 0:
        return {
            'change': float('nan'),
            'change_ci': (float('nan'), float('nan')),
            'efficiency': float('nan'),
            'efficiency_ci': (float('nan'), float('nan')),
        }
    baseline_mean = float(np.mean(baseline_values))
    scenario_mean = float(np.mean(scenario_values))
    change = (scenario_mean / baseline_mean - 1.0) * 100.0
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, baseline_values.size, size=(n_boot, baseline_values.size))
    boot_baseline = baseline_values[idx].mean(axis=1)
    boot_scenario = scenario_values[idx].mean(axis=1)
    boot_change = (boot_scenario / boot_baseline - 1.0) * 100.0
    change_lower, change_upper = np.percentile(boot_change, [2.5, 97.5])
    result = {
        'change': change,
        'change_ci': (float(change_lower), float(change_upper)),
    }
    if intensity is not None:
        efficiency = -change / intensity
        boot_efficiency = -boot_change / intensity
        eff_lower, eff_upper = np.percentile(boot_efficiency, [2.5, 97.5])
        result.update({
            'efficiency': efficiency,
            'efficiency_ci': (float(eff_lower), float(eff_upper)),
        })
    return result


def ci_error_array(values, lowers, uppers):
    values = np.asarray(values, dtype=float)
    lowers = np.asarray(lowers, dtype=float)
    uppers = np.asarray(uppers, dtype=float)
    invalid = (lowers > values) | (uppers < values)
    if np.any(invalid):
        raise ValueError('CI bounds must bracket point estimates.')
    return np.vstack([
        values - lowers,
        uppers - values,
    ])


def _auc_by_run_conspiracy(df_diffusion):
    auc_records = []
    for (run, consp), group in df_diffusion.groupby(['RUN', 'conspiracy']):
        group = group.sort_values('t')
        auc = np.trapezoid(group['adoption_fraction'].values, group['t'].values)
        auc_records.append({'RUN': run, 'conspiracy': consp, 'auc': auc})
    return pd.DataFrame(auc_records)


def total_auc_by_run(df_diffusion):
    df_auc = _auc_by_run_conspiracy(df_diffusion)
    return df_auc.groupby('RUN')['auc'].sum().sort_index()


def paired_auc_reduction_summary(baseline_by_run, scenario_by_run):
    common_runs = baseline_by_run.index.intersection(scenario_by_run.index)
    if len(common_runs) != len(baseline_by_run) or len(common_runs) != len(scenario_by_run):
        raise ValueError('Baseline and scenario run ids must match for paired bootstrap.')
    summary = bootstrap_auc_change_ci(
        baseline_by_run.loc[common_runs].values,
        scenario_by_run.loc[common_runs].values,
    )
    reduction = -summary['change']
    lower = -summary['change_ci'][1]
    upper = -summary['change_ci'][0]
    return reduction, lower, upper


def compute_adoption_auc(df_diffusion):
    """Compute AUC of adoption curves per conspiracy per run, then aggregate."""
    df_auc = _auc_by_run_conspiracy(df_diffusion)

    per_consp_records = []
    for consp, group in df_auc.groupby('conspiracy'):
        mean, lower, upper = bootstrap_mean_ci(group['auc'].values)
        per_consp_records.append({
            'conspiracy': consp,
            'mean_auc': mean,
            'ci_lower': lower,
            'ci_upper': upper,
        })
    per_consp = pd.DataFrame(per_consp_records).sort_values('mean_auc', ascending=False)

    total_per_run = df_auc.groupby('RUN')['auc'].sum().sort_index()
    total_auc = bootstrap_mean_ci(total_per_run.values)

    return per_consp, total_auc


# Compute AUC for all scenarios
scenarios_auc = {}
scenarios_total_by_run = {}
for name, df_diff in [('baseline', df_baseline),
                       ('quarantine_4h',   df_quarantine_4h),
                       ('quarantine_12h',  df_quarantine_12h),
                       ('quarantine_24h',  df_quarantine_24h),
                       ('quarantine_168h', df_quarantine_168h),
                       ('no_temporal_effects', df_no_temporal)]:
    per_consp, total = compute_adoption_auc(df_diff)
    scenarios_auc[name] = (per_consp, total)
    scenarios_total_by_run[name] = total_auc_by_run(df_diff)
    print(f"\n{name}: Total AUC = {total[0]:.2f} (95% simulation run bootstrap interval {total[1]:.2f}, {total[2]:.2f})")

# Build comparison table: per conspiracy AUC across scenarios
auc_comparison = pd.DataFrame({'conspiracy': scenarios_auc['baseline'][0]['conspiracy'].values})
for name, (per_consp, _) in scenarios_auc.items():
    merged = per_consp[['conspiracy', 'mean_auc']].rename(columns={'mean_auc': name})
    auc_comparison = auc_comparison.merge(merged, on='conspiracy')

# Add total row
total_row = {'conspiracy': 'TOTAL'}
for name, (_, total) in scenarios_auc.items():
    total_row[name] = total[0]
auc_comparison = pd.concat([auc_comparison, pd.DataFrame([total_row])], ignore_index=True)

print("\n\nAUC Comparison Across Scenarios:")
print(auc_comparison.to_string(index=False, float_format='%.2f'))


In [ ]:
# Bar chart: per conspiracy AUC across scenarios
scenario_names = list(scenarios_auc.keys())
conspiracies_sorted = scenarios_auc['baseline'][0]['conspiracy'].values

x = np.arange(len(conspiracies_sorted))
width = 0.8 / len(scenario_names)

fig, ax = plt.subplots(figsize=(14, 6))
for i, name in enumerate(scenario_names):
    per_consp = scenarios_auc[name][0].set_index('conspiracy')
    vals = np.asarray([per_consp.loc[c, 'mean_auc'] for c in conspiracies_sorted], dtype=float)
    lowers = np.asarray([per_consp.loc[c, 'ci_lower'] for c in conspiracies_sorted], dtype=float)
    uppers = np.asarray([per_consp.loc[c, 'ci_upper'] for c in conspiracies_sorted], dtype=float)
    errs = ci_error_array(vals, lowers, uppers)
    ax.bar(x + i * width, vals, width, yerr=errs, label=name, capsize=2)

ax.set_xticks(x + width * (len(scenario_names) - 1) / 2)
ax.set_xticklabels([c.replace('ConsProb_', '') for c in conspiracies_sorted],
                   rotation=45, ha='right')
ax.set_ylabel('AUC (adoption fraction x hours)')
ax.set_title('Intervention Effectiveness: Area Under Adoption Curves')
ax.legend()
plt.tight_layout()
fig.savefig('../figures/simulations/fig_02_auc_per_conspiracy.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Total AUC comparison bar chart
scenario_labels = list(scenarios_auc.keys())
total_means = [scenarios_auc[s][1][0] for s in scenario_labels]
total_lowers = [scenarios_auc[s][1][1] for s in scenario_labels]
total_uppers = [scenarios_auc[s][1][2] for s in scenario_labels]
baseline_total_by_run = scenarios_total_by_run['baseline']

pct_reductions = []
for label in scenario_labels:
    if label == 'baseline':
        pct_reductions.append(0.0)
    else:
        reduction, _, _ = paired_auc_reduction_summary(
            baseline_total_by_run,
            scenarios_total_by_run[label],
        )
        pct_reductions.append(reduction)

# Sort by AUC descending
order = np.argsort(total_means)[::-1]
labels_sorted = [scenario_labels[i] for i in order]
means_sorted = np.asarray([total_means[i] for i in order], dtype=float)
lowers_sorted = np.asarray([total_lowers[i] for i in order], dtype=float)
uppers_sorted = np.asarray([total_uppers[i] for i in order], dtype=float)
pcts_sorted = [pct_reductions[i] for i in order]
xerr_sorted = ci_error_array(means_sorted, lowers_sorted, uppers_sorted)

colors = ['#2196F3' if l == 'baseline' else '#FF9800' for l in labels_sorted]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(range(len(labels_sorted)), means_sorted, xerr=xerr_sorted,
               color=colors, capsize=4, edgecolor='white')

for i, (bar, pct) in enumerate(zip(bars, pcts_sorted)):
    label = f'{means_sorted[i]:.1f}'
    if labels_sorted[i] != 'baseline':
        label += f'  ({pct:.1f}% reduction)'
    ax.text(uppers_sorted[i] + 2, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted)
ax.set_xlabel('Total AUC (adoption fraction x hours)')
ax.set_title('Counterfactual Impact: Total Adoption AUC')
ax.set_xlim(0, max(uppers_sorted) * 1.35)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig('../figures/simulations/fig_03_auc_total_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Compare scenarios: adoption summary
all_results = {
    'baseline': results_baseline,
    'quarantine_4h':   results_quarantine_4h,
    'quarantine_12h':  results_quarantine_12h,
    'quarantine_24h':  results_quarantine_24h,
    'quarantine_168h': results_quarantine_168h,
    'no_temporal_effects': results_no_temporal,
}
df_comparison = compare_scenarios(all_results)

print("Scenario Comparison (mean across runs):")
summary = df_comparison.groupby('scenario').agg({
    'total_adoptions': ['mean', 'std'],
    'nodes_with_any_adoption': ['mean', 'std'],
    'mean_conspiracies_per_node': ['mean', 'std'],
    'adoption_rate': ['mean', 'std'],
}).round(4)
print(summary)


## 7. First Adoption Distribution and Empirical Comparison

In [ ]:
# First adoption distribution for all scenarios
for label, res in [('Baseline', results_baseline),
                   ('Quarantine (4h)',   results_quarantine_4h),
                   ('Quarantine (12h)',  results_quarantine_12h),
                   ('Quarantine (24h)',  results_quarantine_24h),
                   ('Quarantine (168h)', results_quarantine_168h),
                   ('No Temporal Effects', results_no_temporal)]:
    df_stats, _ = compute_first_adoption_distribution(res)
    print(f"{label}: First Adoption Statistics (Entry Points):")
    print(df_stats.to_string(index=False))
    print()


In [ ]:
# Compute empirical first-adoption distribution from the graph
first_adoption_counts = {c: 0 for c in CONSPIRACIES}
total_active = 0

for node in G_empirical.nodes():
    first_times = {}
    for consp in CONSPIRACIES:
        activations = G_empirical.nodes[node].get(consp, [])
        if activations:
            first_times[consp] = min(activations)
    if first_times:
        first_consp = min(first_times, key=first_times.get)
        first_adoption_counts[first_consp] += 1
        total_active += 1

empirical_portions = {c: first_adoption_counts[c] / total_active for c in CONSPIRACIES}

print(f"Empirical first-adoption distribution ({total_active} active users):")
for c, p in sorted(empirical_portions.items(), key=lambda x: -x[1]):
    print(f"  {c}: {p:.5f}")

# R² for all scenarios
all_scenario_metrics = {}
for label, res in [('Baseline', results_baseline),
                   ('Quarantine (4h)',   results_quarantine_4h),
                   ('Quarantine (12h)',  results_quarantine_12h),
                   ('Quarantine (24h)',  results_quarantine_24h),
                   ('Quarantine (168h)', results_quarantine_168h),
                   ('No Temporal Effects', results_no_temporal)]:
    metrics = compute_empirical_comparison(res, empirical_portions)
    all_scenario_metrics[label] = metrics
    _, raw_frequencies = compute_first_adoption_distribution(res)
    per_run_metrics = []
    for run_id, group in raw_frequencies.groupby('run_id'):
        run_frequencies = dict(zip(group['conspiracy'], group['frequency']))
        shared = sorted(empirical_portions)
        sim_array = np.asarray([run_frequencies.get(c, 0.0) for c in shared])
        emp_array = np.asarray([empirical_portions[c] for c in shared])
        ss_res = np.sum((emp_array - sim_array) ** 2)
        ss_tot = np.sum((emp_array - emp_array.mean()) ** 2)
        per_run_metrics.append({
            'r_squared': float(1 - ss_res / ss_tot),
            'rmse': float(np.sqrt(np.mean((sim_array - emp_array) ** 2))),
            'correlation': float(np.corrcoef(sim_array, emp_array)[0, 1]),
        })
    scenario_key = label.lower().replace(' ', '_').replace('(', '').replace(')', '')
    validation_metrics[f'first_adoption_{scenario_key}'] = {
        metric: {
            'per_run': [run[metric] for run in per_run_metrics],
            'summary': summarize_run_metric([run[metric] for run in per_run_metrics]),
        }
        for metric in ('r_squared', 'rmse', 'correlation')
    }
    print(f"\n{label} vs. Empirical:")
    print(f"  R² = {metrics['r_squared']:.4f}")
    print(f"  Correlation = {metrics['correlation']:.4f}")
    print(f"  RMSE = {metrics['rmse']:.4f}")
    for metric, values in validation_metrics[f'first_adoption_{scenario_key}'].items():
        interval = values['summary']
        print(f"  Per-run {metric}: median {interval['median']:.4f}, 95% interval [{interval['p2_5']:.4f}, {interval['p97_5']:.4f}]")


In [ ]:
# Plot empirical vs. simulated first-adoption comparison for all scenarios
scenario_plot_data = [
    ('Baseline', results_baseline),
    ('Quarantine (4h)',   results_quarantine_4h),
    ('Quarantine (12h)',  results_quarantine_12h),
    ('Quarantine (24h)',  results_quarantine_24h),
    ('Quarantine (168h)', results_quarantine_168h),
    ('No Temporal Effects', results_no_temporal),
]

n_scenarios = len(scenario_plot_data)
fig, axes = plt.subplots(1, n_scenarios, figsize=(5 * n_scenarios, 5))

for ax, (name, res) in zip(axes, scenario_plot_data):
    df_stats, _ = compute_first_adoption_distribution(res)
    metrics = all_scenario_metrics[name]

    shared = sorted(set(df_stats['conspiracy']) & set(empirical_portions.keys()))
    sim_vals = [df_stats[df_stats['conspiracy'] == c]['mean_frequency'].iloc[0] for c in shared]
    emp_vals = [empirical_portions[c] for c in shared]
    labels = [c.replace('ConsProb_', '') for c in shared]

    ax.scatter(emp_vals, sim_vals, s=40)
    for i, label in enumerate(labels):
        ax.annotate(label, (emp_vals[i], sim_vals[i]), fontsize=7,
                    xytext=(3, 3), textcoords='offset points')

    # 45-degree line
    lim = max(max(emp_vals), max(sim_vals)) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel('Empirical Proportion')
    ax.set_ylabel('Simulated Proportion')
    ax.set_title(f'{name}\n(R²={metrics["r_squared"]:.3f})')
    ax.set_aspect('equal')

plt.tight_layout()
fig.savefig('../figures/simulations/fig_04_first_adoption_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 8. Co-Adoption Pattern Validation

Do conspiracies that co-occur within the same individuals in the empirical data also co-occur in the simulation? We compare pairwise Jaccard similarity matrices between empirical and simulated adoption patterns.

In [ ]:
# 1. Empirical co-adoption matrix (Jaccard similarity)
empirical_coadoption = compute_coadoption_matrix(G_empirical, CONSPIRACIES)

# 2. Simulated co-adoption matrix (averaged across baseline runs)
def compute_simulated_coadoption(scenario_results, conspiracies):
    """Compute Jaccard co-adoption matrix from simulation results, averaged across runs."""
    n = len(conspiracies)
    matrices = []

    for run in scenario_results.runs:
        # Build adopter sets per conspiracy from adoption_histories
        adopters = {c: set() for c in conspiracies}
        for node_id, hist in run.adoption_histories.items():
            for c in hist:
                if c in adopters:
                    adopters[c].add(node_id)

        # Compute Jaccard matrix
        mat = np.ones((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                set_i = adopters[conspiracies[i]]
                set_j = adopters[conspiracies[j]]
                union = len(set_i | set_j)
                jaccard = len(set_i & set_j) / union if union > 0 else 0.0
                mat[i, j] = jaccard
                mat[j, i] = jaccard
        matrices.append(mat)

    mean_mat = np.mean(matrices, axis=0)
    return (
        pd.DataFrame(mean_mat, index=conspiracies, columns=conspiracies),
        np.asarray(matrices),
    )

simulated_coadoption, simulated_coadoption_runs = compute_simulated_coadoption(
    results_baseline, CONSPIRACIES
)

# 3. Correlation and R² between upper triangles
triu_idx = np.triu_indices(len(CONSPIRACIES), k=1)
emp_values = empirical_coadoption.values[triu_idx]
sim_values = simulated_coadoption.values[triu_idx]

r = float(np.corrcoef(emp_values, sim_values)[0, 1])

# R² (coefficient of determination): measures absolute prediction accuracy
ss_res = np.sum((emp_values - sim_values) ** 2)
ss_tot = np.sum((emp_values - emp_values.mean()) ** 2)
r2_coadoption = 1 - ss_res / ss_tot
rmse_coadoption = float(np.sqrt(np.mean((emp_values - sim_values) ** 2)))

coadoption_run_metrics = []
for run_matrix in simulated_coadoption_runs:
    run_values = run_matrix[triu_idx]
    run_ss_res = np.sum((emp_values - run_values) ** 2)
    coadoption_run_metrics.append({
        'r_squared': float(1 - run_ss_res / ss_tot),
        'rmse': float(np.sqrt(np.mean((emp_values - run_values) ** 2))),
        'correlation': float(np.corrcoef(emp_values, run_values)[0, 1]),
    })
validation_metrics['coadoption'] = {
        metric: {
            'per_run': [run[metric] for run in coadoption_run_metrics],
            'summary': summarize_run_metric([run[metric] for run in coadoption_run_metrics]),
        }
        for metric in ('r_squared', 'rmse', 'correlation')
}

print(f"Co-adoption Jaccard similarity (empirical vs. simulated):")
print(f"  R² (coefficient of determination) = {r2_coadoption:.4f}")
print(f"  Pearson r = {r:.4f}")
print(f"  RMSE = {rmse_coadoption:.4f}")
for metric, values in validation_metrics['coadoption'].items():
    interval = values['summary']
    print(f"  Per-run {metric}: median {interval['median']:.4f}, 95% interval [{interval['p2_5']:.4f}, {interval['p97_5']:.4f}]")
print(f"  Mean empirical Jaccard: {emp_values.mean():.4f}")
print(f"  Mean simulated Jaccard: {sim_values.mean():.4f}")


In [ ]:
# Side-by-side heatmaps
short_labels = [c.replace('ConsProb_', '') for c in CONSPIRACIES]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Empirical
sns.heatmap(empirical_coadoption.values, ax=axes[0], vmin=0, vmax=1,
            xticklabels=short_labels, yticklabels=short_labels,
            cmap='YlOrRd', annot=True, fmt='.2f', annot_kws={'size': 7})
axes[0].set_title('Empirical Co-Adoption (Jaccard)')
axes[0].tick_params(axis='both', labelsize=8)

# Simulated
sns.heatmap(simulated_coadoption.values, ax=axes[1], vmin=0, vmax=1,
            xticklabels=short_labels, yticklabels=short_labels,
            cmap='YlOrRd', annot=True, fmt='.2f', annot_kws={'size': 7})
axes[1].set_title('Simulated Co-Adoption (Jaccard, baseline)')
axes[1].tick_params(axis='both', labelsize=8)

# Scatter plot: empirical vs simulated per conspiracy pair
axes[2].scatter(emp_values, sim_values, s=30, alpha=0.6)
lim = max(emp_values.max(), sim_values.max()) * 1.1
axes[2].plot([0, lim], [0, lim], 'k--', alpha=0.3)
axes[2].set_xlim(0, lim)
axes[2].set_ylim(0, lim)
axes[2].set_xlabel('Empirical Jaccard')
axes[2].set_ylabel('Simulated Jaccard')
axes[2].set_title(f'Pairwise Comparison (r = {r:.3f})')
axes[2].set_aspect('equal')

plt.tight_layout()
fig.savefig('../figures/simulations/fig_05_coadoption_jaccard.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Adoption Breadth Validation

How many distinct conspiracies does each user adopt? If the ABM reproduces the
empirical adoption breadth distribution, it confirms the model captures not just
*which* conspiracies spread but also *how deeply* users engage with multiple theories.

In [ ]:
from scipy.spatial.distance import jensenshannon

# --- 1. Empirical distribution: conspiracies adopted per user ---
max_count = len(CONSPIRACIES)
empirical_counts = np.zeros(max_count + 1)

for node in G_empirical.nodes():
    n_adopted = 0
    for c in CONSPIRACIES:
        acts = G_empirical.nodes[node].get(c, [])
        if acts and not all(np.isnan(a) for a in acts):
            n_adopted += 1
    empirical_counts[n_adopted] += 1

n_empirical_users = G_empirical.number_of_nodes()
empirical_shares = empirical_counts / n_empirical_users

# --- 2. Simulated distribution (averaged across baseline runs) ---
sim_shares_all_runs = []

for run in results_baseline.runs:
    sim_counts = np.zeros(max_count + 1)

    for node_id, hist in run.adoption_histories.items():
        n_adopted = sum(1 for c in hist if c in set(CONSPIRACIES))
        n_adopted = min(n_adopted, max_count)
        sim_counts[n_adopted] += 1

    # Nodes NOT in adoption_histories adopted 0
    sim_counts[0] += run.n_nodes - len(run.adoption_histories)
    sim_shares_all_runs.append(sim_counts / run.n_nodes)

sim_shares_mean = np.mean(sim_shares_all_runs, axis=0)
sim_shares_std = np.std(sim_shares_all_runs, axis=0)

# --- 3. Grouped bar chart ---
x = np.arange(max_count + 1)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: Full distribution
ax = axes[0]
ax.bar(x - width / 2, empirical_shares, width, label='Empirical',
       color='steelblue', alpha=0.8)
ax.bar(x + width / 2, sim_shares_mean, width, label='Simulated (baseline)',
       color='coral', alpha=0.8, yerr=sim_shares_std, capsize=2)
ax.set_xlabel('Number of conspiracies adopted')
ax.set_ylabel('Share of users')
ax.set_title('Adoption Breadth Distribution')
ax.set_xticks(x)
ax.legend()
ax.set_xlim(-0.5, max_count + 0.5)

# Panel B: Zoomed to adopters only
ax2 = axes[1]
x_zoom = np.arange(1, max_count + 1)
ax2.bar(x_zoom - width / 2, empirical_shares[1:], width, label='Empirical',
        color='steelblue', alpha=0.8)
ax2.bar(x_zoom + width / 2, sim_shares_mean[1:], width, label='Simulated (baseline)',
        color='coral', alpha=0.8, yerr=sim_shares_std[1:], capsize=2)
ax2.set_xlabel('Number of conspiracies adopted')
ax2.set_ylabel('Share of users')
ax2.set_title('Adoption Breadth (adopters only)')
ax2.set_xticks(x_zoom)
ax2.legend()

plt.tight_layout()
fig.savefig('../figures/simulations/fig_06_adoption_breadth.pdf', dpi=300, bbox_inches='tight')
plt.show()

# --- 4. Statistical comparison ---
# Jensen-Shannon distance is computed for every simulation run.
jsd = float(jensenshannon(empirical_shares, sim_shares_mean))
jsd_values = [
    float(jensenshannon(empirical_shares, sim_shares_run))
    for sim_shares_run in sim_shares_all_runs
]
validation_metrics['adoption_breadth_jsd'] = {
    'per_run': jsd_values,
    'summary': summarize_run_metric(jsd_values),
}

# NOTE: Mean breadth is not reported here because it is mechanically identical
# between empirical and simulated data (an arithmetic consequence of the
# target_total_adoptions early stopping constraint, not a model achievement).

print(f"Adoption breadth distribution comparison:")
print(f"  Jensen-Shannon distance for mean distribution: {jsd:.4f}")
jsd_summary = validation_metrics['adoption_breadth_jsd']['summary']
print(f"  Per-run Jensen-Shannon distance: median {jsd_summary['median']:.4f}, 95% interval [{jsd_summary['p2_5']:.4f}, {jsd_summary['p97_5']:.4f}]")
print(f"  Share adopting 0: empirical {empirical_shares[0]:.4f}, simulated {sim_shares_mean[0]:.4f}")
print(f"  Share adopting 1+: empirical {1 - empirical_shares[0]:.4f}, simulated {1 - sim_shares_mean[0]:.4f}")
print(f"")
print(f"  NOTE: The s7 > 0 design choice requires network exposure for adoption,")
print(f"  producing fewer non-adopters (simulated {sim_shares_mean[0]:.1%} vs empirical {empirical_shares[0]:.1%}).")


### Per-Conspiracy Adoption Share

What fraction of users adopted each conspiracy? Unlike the first-adoption
distribution (Section 7), which only counts entry points, this compares the
total adoption prevalence of each conspiracy.

In [ ]:
# --- 1. Empirical per-conspiracy adoption share ---
n_total_empirical = G_empirical.number_of_nodes()
empirical_adoption_share = {}

for c in CONSPIRACIES:
    n_adopters = 0
    for node in G_empirical.nodes():
        acts = G_empirical.nodes[node].get(c, [])
        if acts and not all(np.isnan(a) for a in acts):
            n_adopters += 1
    empirical_adoption_share[c] = n_adopters / n_total_empirical

# --- 2. Simulated per-conspiracy adoption share (averaged across runs) ---
sim_adoption_shares = {c: [] for c in CONSPIRACIES}

for run in results_baseline.runs:
    adopter_counts = {c: 0 for c in CONSPIRACIES}
    for node_id, hist in run.adoption_histories.items():
        for c in hist:
            if c in adopter_counts:
                adopter_counts[c] += 1
    for c in CONSPIRACIES:
        sim_adoption_shares[c].append(adopter_counts[c] / run.n_nodes)

sim_share_mean = {c: np.mean(sim_adoption_shares[c]) for c in CONSPIRACIES}
sim_share_std = {c: np.std(sim_adoption_shares[c]) for c in CONSPIRACIES}

# Sort by empirical share (descending)
sorted_cons = sorted(CONSPIRACIES, key=lambda c: -empirical_adoption_share[c])
short_names = [c.replace('ConsProb_', '') for c in sorted_cons]
emp_vals = [empirical_adoption_share[c] for c in sorted_cons]
sim_vals = [sim_share_mean[c] for c in sorted_cons]
sim_errs = [sim_share_std[c] for c in sorted_cons]

# --- 3. Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Panel A: Grouped bar chart
x = np.arange(len(sorted_cons))
width = 0.35
axes[0].bar(x - width / 2, emp_vals, width, label='Empirical',
            color='steelblue', alpha=0.8)
axes[0].bar(x + width / 2, sim_vals, width, label='Simulated (baseline)',
            color='coral', alpha=0.8, yerr=sim_errs, capsize=2)
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Share of users who adopted')
axes[0].set_title('Per-Conspiracy Adoption Prevalence')
axes[0].legend()

# Panel B: Scatter plot with error bars and 45-degree line
axes[1].errorbar(emp_vals, sim_vals, yerr=sim_errs, fmt='o', capsize=3,
                 color='steelblue', alpha=0.7, markersize=6)
for i, name in enumerate(short_names):
    axes[1].annotate(name, (emp_vals[i], sim_vals[i]), fontsize=7,
                     xytext=(3, 3), textcoords='offset points')
lim = max(max(emp_vals), max(sim_vals)) * 1.1
axes[1].plot([0, lim], [0, lim], 'k--', alpha=0.3)
axes[1].set_xlim(0, lim)
axes[1].set_ylim(0, lim)
axes[1].set_xlabel('Empirical adoption share')
axes[1].set_ylabel('Simulated adoption share')

# R² (coefficient of determination)
ss_res = np.sum((np.array(sim_vals) - np.array(emp_vals)) ** 2)
ss_tot = np.sum((np.array(emp_vals) - np.mean(emp_vals)) ** 2)
r2_share = 1 - ss_res / ss_tot
rmse_share = float(np.sqrt(np.mean((np.array(sim_vals) - np.array(emp_vals)) ** 2)))
r_share = float(np.corrcoef(emp_vals, sim_vals)[0, 1])
narrative_share_run_metrics = []
for run_index in range(results_baseline.n_runs):
    run_values = np.asarray([sim_adoption_shares[c][run_index] for c in sorted_cons])
    run_ss_res = np.sum((np.asarray(emp_vals) - run_values) ** 2)
    narrative_share_run_metrics.append({
        'r_squared': float(1 - run_ss_res / ss_tot),
        'rmse': float(np.sqrt(np.mean((np.asarray(emp_vals) - run_values) ** 2))),
        'correlation': float(np.corrcoef(emp_vals, run_values)[0, 1]),
    })
validation_metrics['narrative_shares'] = {
    metric: {
        'per_run': [run[metric] for run in narrative_share_run_metrics],
        'summary': summarize_run_metric([run[metric] for run in narrative_share_run_metrics]),
    }
    for metric in ('r_squared', 'rmse', 'correlation')
}
axes[1].set_title(f'Per-Conspiracy Comparison (R² = {r2_share:.3f})')
axes[1].set_aspect('equal')

plt.tight_layout()
fig.savefig('../figures/simulations/fig_07_per_conspiracy_prevalence.pdf', dpi=300, bbox_inches='tight')
plt.show()

# --- 4. Print summary ---
print(f"Per-conspiracy adoption share (R² = {r2_share:.4f}, Pearson r = {r_share:.4f}, RMSE = {rmse_share:.4f}):\n")
for metric, values in validation_metrics['narrative_shares'].items():
    interval = values['summary']
    print(f"Per-run {metric}: median {interval['median']:.4f}, 95% interval [{interval['p2_5']:.4f}, {interval['p97_5']:.4f}]")
print()
print(f"{'Conspiracy':<16} {'Empirical':>10} {'Simulated':>10} {'Diff':>10}")
print("-" * 48)
for c in sorted_cons:
    e = empirical_adoption_share[c]
    s = sim_share_mean[c]
    name = c.replace('ConsProb_', '')
    print(f"{name:<16} {e:>10.4f} {s:>10.4f} {s - e:>+10.4f}")


## 9. Summary

The revised agent-based model fixes critical issues from the original ABM:

| Issue | Old ABM | New ABM |
|---|---|---|
| Model dispatch | 2 hazard functions | Models 1-5 based on adoption count |
| Exposure window | 168h (7-day) | 336h (14-day) rolling window |
| Cross-cluster | Never computed | From agent adoption history |
| Hazard→probability | P = min(1, h) | P = 1 - exp(-h) |
| Hawkes re-sharing | Simplified: adopt once → forever active | Full Hawkes self-exciting process |

The baseline scenario should reproduce empirical diffusion patterns, while the
quarantine counterfactuals test whether cooling-off periods after adoption would
slow cross-conspiracy spread.

## 10. Hawkes Model Comparison and Simulation Check

Summarize the relative model fit and compare observed repeated sharing with a matched Hawkes simulation. Two diagnostics:

1. **Model Comparison (AIC)**: Hawkes versus homogeneous Poisson and inhomogeneous Poisson.
2. **Empirical versus Simulated**: Compare interevent time distributions between data and Hawkes simulated sequences.

### 10.1 Pooled Diagnostics

In [ ]:
from conspiracy_analysis.visualization import plot_hawkes_goodness_of_fit
from conspiracy_analysis.models.hawkes_sharing import hawkes_model_comparison

# Run diagnostics and display two panel figure
fig = plot_hawkes_goodness_of_fit(
    sequences, hawkes_fit, observation_ends=observation_ends
)

# Print numerical summaries
print("=== Model Comparison (AIC / BIC) ===")
comp = hawkes_model_comparison(sequences, hawkes_fit, observation_ends=observation_ends)
print(comp.to_string(index=False))
print(f"\n  Best model by AIC: {comp.loc[comp['AIC'].idxmin(), 'model']}")
print(f"  ΔAIC (Hawkes vs Poisson): {comp.loc[comp['model']=='Homogeneous Poisson', 'AIC'].values[0] - comp.loc[comp['model']=='Hawkes', 'AIC'].values[0]:.0f}")

## 11. Content Moderation Counterfactual

What if a platform's content moderation algorithm could detect and suppress conspiracy tweets? Each re-share event has probability `block_rate` of being flagged and shadow banned from the network. Blocked shares still contribute to the poster's own Hawkes self-excitation (they see their own post and keep posting) but are invisible to followers (not counted in s7 exposure).

Each tweet, including the user's first conspiracy sharing event, is subject to the same detection probability.

In [ ]:
# Run content moderation scenarios at different detection accuracies
moderation_results = {}

for rate in MODERATION_RATES:
    scenario_mod = content_moderation(block_rate=rate)
    print(f"Running {scenario_mod.name}...")

    results_mod = run_scenario(G, sim_config, scenario_mod, runs=RUNS, n_jobs=SIM_N_JOBS)
    moderation_results[rate] = results_mod

    total_adoptions = np.mean([
        sum(len(h) for h in run.adoption_histories.values())
        for run in results_mod.runs
    ])
    print(f"  Mean total adoptions: {total_adoptions:.0f}")

print(f"\nCompleted {len(moderation_results)} moderation scenarios.")

In [ ]:
# Dose response curve: detection accuracy vs. adoption reduction
baseline_total_by_run = scenarios_total_by_run['baseline']
baseline_auc_total, baseline_auc_lower, baseline_auc_upper = bootstrap_mean_ci(baseline_total_by_run.values)

mod_rates = []
mod_auc_means = []
mod_auc_lowers = []
mod_auc_uppers = []
mod_pct_reductions = []
mod_pct_reduction_lowers = []
mod_pct_reduction_uppers = []

for rate in sorted(moderation_results.keys()):
    df_mod = compute_diffusion_curves(moderation_results[rate], steps=counterfactual_steps)
    mod_by_run = total_auc_by_run(df_mod)
    auc_mean, auc_lower, auc_upper = bootstrap_mean_ci(mod_by_run.values)
    pct_reduction, pct_lower, pct_upper = paired_auc_reduction_summary(
        baseline_total_by_run,
        mod_by_run,
    )

    mod_rates.append(rate)
    mod_auc_means.append(auc_mean)
    mod_auc_lowers.append(auc_lower)
    mod_auc_uppers.append(auc_upper)
    mod_pct_reductions.append(pct_reduction)
    mod_pct_reduction_lowers.append(pct_lower)
    mod_pct_reduction_uppers.append(pct_upper)

    print(f"  Block {rate*100:5.1f}%: AUC = {auc_mean:.1f} "
          f"(95% simulation run bootstrap interval {auc_lower:.1f}, {auc_upper:.1f})  "
          f"({pct_reduction:+.1f}% vs baseline)")

mod_auc_errors = ci_error_array(mod_auc_means, mod_auc_lowers, mod_auc_uppers)
mod_pct_errors = ci_error_array(mod_pct_reductions, mod_pct_reduction_lowers, mod_pct_reduction_uppers)

# Plot dose response
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: AUC vs block rate
ax1.errorbar([r * 100 for r in mod_rates], mod_auc_means, yerr=mod_auc_errors,
             fmt='o-', capsize=4, color='#d32f2f', label='Content moderation')
ax1.axhline(baseline_auc_total, color='#2196F3', linestyle='--', label='Baseline (no moderation)')
ax1.fill_between([0, 100],
                 baseline_auc_lower,
                 baseline_auc_upper,
                 alpha=0.15, color='#2196F3')
ax1.set_xlabel('Detection Accuracy (%)')
ax1.set_ylabel('Total AUC (adoption fraction x hours)')
ax1.set_title('Content Moderation: Dose Response')
ax1.legend()
ax1.set_xlim(40, 100)

# Right: percent reduction vs block rate
bars = ax2.bar([f'{r*100:.0f}%' for r in mod_rates], mod_pct_reductions,
               yerr=mod_pct_errors, capsize=4,
               color='#d32f2f', edgecolor='white')
ax2.set_xlabel('Detection Accuracy')
ax2.set_ylabel('% Reduction in Total AUC vs Baseline')
ax2.set_title('Effectiveness of Content Moderation')
for i, pct in enumerate(mod_pct_reductions):
    ax2.text(i, pct + mod_pct_errors[1][i] + 0.5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig.savefig('../figures/simulations/fig_08_content_moderation_dose_response.pdf', dpi=300, bbox_inches='tight')
plt.show()


## 12. Reputation Nudge Counterfactual

What if the platform warned users before they share a conspiracy for the first time? When a user is about to adopt a conspiracy through peer influence, a prompt warns them that sharing this content may harm their reputation. With a configurable probability, the user heeds the warning: no adoption occurs and they become permanently immune to that conspiracy.

Seeds are not affected — only organic (socially-driven) adoptions face the nudge. This tests how effective reputation-based friction is at slowing conspiracy adoption.

In [ ]:
from conspiracy_analysis.simulation.scenarios import reputation_nudge

NUDGE_RATES = [0.05, 0.1, 0.2, 0.3, 0.5]

nudge_results = {}

for rate in NUDGE_RATES:
    scenario_nudge = reputation_nudge(rejection_rate=rate)
    print(f"Running {scenario_nudge.name}...")

    results_nudge = run_scenario(G, sim_config, scenario_nudge, runs=RUNS, n_jobs=SIM_N_JOBS)
    nudge_results[rate] = results_nudge

    total_adoptions = np.mean([
        sum(len(h) for h in run.adoption_histories.values())
        for run in results_nudge.runs
    ])
    print(f"  Mean total adoptions: {total_adoptions:.0f}")

print(f"\nCompleted {len(nudge_results)} nudge scenarios.")

In [ ]:
# Dose response curve: nudge rejection rate vs. adoption reduction
baseline_total_by_run = scenarios_total_by_run['baseline']
baseline_auc_total, baseline_auc_lower, baseline_auc_upper = bootstrap_mean_ci(baseline_total_by_run.values)

nudge_rates_pct = []
nudge_auc_means = []
nudge_auc_lowers = []
nudge_auc_uppers = []
nudge_pct_reductions = []
nudge_pct_reduction_lowers = []
nudge_pct_reduction_uppers = []

for rate in sorted(nudge_results.keys()):
    df_nudge = compute_diffusion_curves(nudge_results[rate], steps=counterfactual_steps)
    nudge_by_run = total_auc_by_run(df_nudge)
    auc_mean, auc_lower, auc_upper = bootstrap_mean_ci(nudge_by_run.values)
    pct_reduction, pct_lower, pct_upper = paired_auc_reduction_summary(
        baseline_total_by_run,
        nudge_by_run,
    )

    nudge_rates_pct.append(rate)
    nudge_auc_means.append(auc_mean)
    nudge_auc_lowers.append(auc_lower)
    nudge_auc_uppers.append(auc_upper)
    nudge_pct_reductions.append(pct_reduction)
    nudge_pct_reduction_lowers.append(pct_lower)
    nudge_pct_reduction_uppers.append(pct_upper)

    print(f"  Nudge {rate*100:5.1f}%: AUC = {auc_mean:.1f} "
          f"(95% simulation run bootstrap interval {auc_lower:.1f}, {auc_upper:.1f})  "
          f"({pct_reduction:+.1f}% vs baseline)")

nudge_auc_errors = ci_error_array(nudge_auc_means, nudge_auc_lowers, nudge_auc_uppers)
nudge_pct_errors = ci_error_array(nudge_pct_reductions, nudge_pct_reduction_lowers, nudge_pct_reduction_uppers)

# Plot dose response
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: AUC vs nudge rate
ax1.errorbar([r * 100 for r in nudge_rates_pct], nudge_auc_means, yerr=nudge_auc_errors,
             fmt='o-', capsize=4, color='#7B1FA2', label='Reputation nudge')
ax1.axhline(baseline_auc_total, color='#2196F3', linestyle='--', label='Baseline (no nudge)')
ax1.fill_between([0, 60],
                 baseline_auc_lower,
                 baseline_auc_upper,
                 alpha=0.15, color='#2196F3')
ax1.set_xlabel('Nudge Rejection Rate (%)')
ax1.set_ylabel('Total AUC (adoption fraction x hours)')
ax1.set_title('Reputation Nudge: Dose Response')
ax1.legend()
ax1.set_xlim(0, 60)

# Right: percent reduction vs nudge rate
bars = ax2.bar([f'{r*100:.0f}%' for r in nudge_rates_pct], nudge_pct_reductions,
               yerr=nudge_pct_errors, capsize=4,
               color='#7B1FA2', edgecolor='white')
ax2.set_xlabel('Nudge Rejection Rate')
ax2.set_ylabel('% Reduction in Total AUC vs Baseline')
ax2.set_title('Effectiveness of Reputation Nudge')
for i, pct in enumerate(nudge_pct_reductions):
    ax2.text(i, pct + nudge_pct_errors[1][i] + 0.5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig.savefig('../figures/simulations/fig_09_reputation_nudge_dose_response.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# Save all simulation results and configuration to a single file
# ============================================================
import os

save_path = intermediate_path('simulation_results_all.pkl')
manifest_path = intermediate_path('simulation_results_all_manifest.json')

def numeric_summary(values):
    values = np.asarray(values, dtype=float)
    return {
        'count': int(len(values)),
        'minimum': float(np.min(values)),
        'mean': float(np.mean(values)),
        'median': float(np.median(values)),
        'maximum': float(np.max(values)),
        'p2_5': float(np.percentile(values, 2.5)),
        'p97_5': float(np.percentile(values, 97.5)),
    }

adjacency_degrees = [G.degree(node) for node in G.nodes()]
degree_summaries = {
    'simulation_adjacency_degree': numeric_summary(adjacency_degrees),
    'simulation_log1p_adjacency_degree': numeric_summary(np.log1p(adjacency_degrees)),
}
model_centering = {
    str(model_number): {
        'baseline_type': spec.baseline.type,
        'baseline_reference': spec.baseline.reference,
        'centering_factor': float(spec.baseline.centering_factor),
    }
    for model_number, spec in sorted(sim_config.cox_models.items())
}
scenario_settings = {
    'baseline': asdict(results_baseline.scenario),
    'quarantine_4h': asdict(results_quarantine_4h.scenario),
    'quarantine_12h': asdict(results_quarantine_12h.scenario),
    'quarantine_24h': asdict(results_quarantine_24h.scenario),
    'quarantine_168h': asdict(results_quarantine_168h.scenario),
    'no_temporal_effects': asdict(results_no_temporal.scenario),
    'content_moderation': {str(rate): asdict(result.scenario) for rate, result in moderation_results.items()},
    'reputation_nudge': {str(rate): asdict(result.scenario) for rate, result in nudge_results.items()},
}
horizon_rule = {
    'baseline': 'stop each run at the empirical total adoption target',
    'counterfactual': 'disable target and use the mean baseline stopping time',
    'counterfactual_steps': int(counterfactual_steps),
    'conservative_approximation': 'Baseline padding and truncation make policy reductions conservative approximations.',
}
input_paths = {
    'graph_cache': anonymized_graph_path(),
    'semantic_clustering': intermediate_path('semantic_clustering.pkl'),
    'conspiracy_config': Path('../config/conspiracies.yaml'),
    'simulation_notebook': Path('02_simulation.ipynb'),
}
for model_number in sorted(cox_models):
    filename = 'model_2a_cox.pkl' if model_number == 2 else f'model_{model_number}_cox.pkl'
    input_paths[f'cox_model_{model_number}'] = intermediate_path(filename)
input_sha256 = {name: sha256_file(path) for name, path in input_paths.items()}
package_versions = collect_package_versions([
    'numpy', 'pandas', 'networkx', 'scipy', 'lifelines', 'joblib', 'matplotlib', 'seaborn'
])

# Acceptance checks required before replacing the aggregate artifact.
assert all(np.isfinite(item['centering_factor']) and item['centering_factor'] > 0 for item in model_centering.values())
assert hawkes_fit.converged and np.isfinite(hawkes_fit.log_likelihood)
assert np.all(np.isfinite(comp['log_likelihood']))
required_results = [
    results_baseline, results_quarantine_4h, results_quarantine_12h,
    results_quarantine_24h, results_quarantine_168h, results_no_temporal,
    *moderation_results.values(), *nudge_results.values(),
]
assert all(result.n_runs == RUNS for result in required_results)
assert counterfactual_steps == results_baseline.mean_steps_completed

manifest = {
    'manifest_version': 'simulation_correction_v2',
    **INITIAL_GIT_STATE,
    'package_versions': package_versions,
    'input_sha256': input_sha256,
    'graph_counts': {
        'full_configured_nodes': int(G_full_undirected.number_of_nodes()),
        'full_configured_undirected_edges': int(G_full_undirected.number_of_edges()),
        'removed_nonhuman_nodes': int(len(nodes_to_remove)),
        'simulation_human_giant_component_nodes': int(G.number_of_nodes()),
        'simulation_human_undirected_edges': int(G.number_of_edges()),
    },
    'degree_summaries': degree_summaries,
    'model_centering': model_centering,
    'hawkes_fit': asdict(hawkes_fit),
    'point_process_comparison': comp.to_dict(orient='records'),
    'scenario_settings': scenario_settings,
    'horizon_rule': horizon_rule,
    'require_peer_exposure': bool(sim_config.require_peer_exposure),
    'degree_source': 'human_simulation_graph',
    'random_seeds': list(range(1, RUNS + 1)),
    'validation_metrics': validation_metrics,
}
manifest = canonicalize_json_value(manifest)

save_bundle = {
    # --- Configuration ---
    'config': {
        'STEPS': STEPS,
        'RUNS': RUNS,
        'SIM_N_JOBS': SIM_N_JOBS,
        'SEED_FRACTION': SEED_FRACTION,
        'EXPOSURE_WINDOW': EXPOSURE_WINDOW,
        'EQUAL_START': EQUAL_START,
        'USE_SYNTHETIC': USE_SYNTHETIC,
        'T_QUARANTINE_4H':   T_QUARANTINE_4H,
        'T_QUARANTINE_12H':  T_QUARANTINE_12H,
        'T_QUARANTINE_24H':  T_QUARANTINE_24H,
        'T_QUARANTINE_168H': T_QUARANTINE_168H,
        'MODERATION_RATES': MODERATION_RATES,
        'NUDGE_RATES': NUDGE_RATES,
        'CONSPIRACIES': CONSPIRACIES,
        'hawkes_params': hawkes_params,
        'hawkes_fit': asdict(hawkes_fit),
        'cluster_assignments': cluster_assignments,
        'entry_times': entry_times,
        'counterfactual_steps': counterfactual_steps,
        'empirical_total_adoptions': empirical_total_adoptions,
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'require_peer_exposure': sim_config.require_peer_exposure,
        'degree_source': 'human_simulation_graph',
        'degree_summaries': degree_summaries,
        'model_centering': model_centering,
        'horizon_rule': horizon_rule,
        'source_commit': INITIAL_GIT_STATE['source_commit'],
    },
    # --- Baseline + standard counterfactuals ---
    'results_baseline': results_baseline,
    'results_quarantine_4h':   results_quarantine_4h,
    'results_quarantine_12h':  results_quarantine_12h,
    'results_quarantine_24h':  results_quarantine_24h,
    'results_quarantine_168h': results_quarantine_168h,
    'results_no_temporal': results_no_temporal,
    # --- Content moderation (keyed by block rate) ---
    'moderation_results': moderation_results,
    # --- Reputation nudge (keyed by rejection rate) ---
    'nudge_results': nudge_results,
    # --- Empirical reference data ---
    'empirical_portions': empirical_portions,
    'empirical_adoption_share': empirical_adoption_share,
    'empirical_shares': empirical_shares,
    'validation_metrics': validation_metrics,
    'manifest': manifest,
}

atomic_joblib_dump(save_bundle, save_path)
atomic_json_dump(manifest, manifest_path)
print(f"Saved all results to {os.path.abspath(save_path)}")
print(f"  Scenarios: baseline, quarantine (4h/12h/24h/168h), no_temporal_effects")
print(f"  Content moderation: {len(moderation_results)} rates ({[f'{r*100:.0f}%' for r in sorted(moderation_results.keys())]})")
print(f"  Reputation nudge: {len(nudge_results)} rates ({[f'{r*100:.0f}%' for r in sorted(nudge_results.keys())]})")
print(f"  {RUNS} runs per scenario, {SIM_N_JOBS} worker cap, {counterfactual_steps} steps")
print(f"  Includes sharing histories (all + visible) per node per run")
